___
<a href='https://cafe.naver.com/jmhonglab'><p style="text-align:center;"><img src='https://lh3.googleusercontent.com/lY3ySXooSmwsq5r-mRi7uiypbo0Vez6pmNoQxMFhl9fmZJkRHu5lO2vo7se_0YOzgmDyJif9fi4_z0o3ZFdwd8NVSWG6Ea80uWaf3pOHpR4GHGDV7kaFeuHR3yAjIJjDgfXMxsvw=w2400'  class="center" width="50%" height="50%"/></p></a>
___
<center><em>Content Copyright by HongLab, Inc.</em></center>

# 얼굴 감지(Face Detection)

[MediaPipe](https://google.github.io/mediapipe/)의 [Face Mesh](https://google.github.io/mediapipe/solutions/face_mesh) 사용  

활용 사례: [Flappybird](https://www.youtube.com/shorts/3ywKjfO24J0), [SparkAR Gallery](https://www.facebook.com/sparkarhub/gallery?ref=spark_website), [운전자 졸음 감지](https://www.youtube.com/watch?v=Q23K7G1gJgY) 등


[Landmark index map](https://github.com/google/mediapipe/blob/a908d668c730da128dfa8d9f6bd25d519d006692/mediapipe/modules/face_geometry/data/canonical_face_model_uv_visualization.png)

In [4]:
import cv2
import mediapipe as mp

# 미디어파이프가 찾아주는 랜드마크 좌표를 우리가 사용하기 편한 이미지스페이스 좌표로 변환
def find_landmarks(img_bgr, face_mesh):
    """Normalized image space to pixel space"""

    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    results = face_mesh.process(img_rgb)
    landmarks = []
    if results.multi_face_landmarks:
        for person in results.multi_face_landmarks:
            face = []
            for id, lm in enumerate(person.landmark):
                ih, iw, _ = img_bgr.shape
                x, y = int(lm.x * iw), int(lm.y * ih)
                face.append([x, y])
            landmarks.append(face)
    return landmarks

#cap = cv2.VideoCapture("jmhong_face.mp4")
cap = cv2.VideoCapture(0)

with mp.solutions.face_mesh.FaceMesh(
    max_num_faces=1,
    refine_landmarks=True,
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5,
) as face_mesh:

    while cap.isOpened():

        success, image = cap.read()
        if not success:            
            break

        landmarks = find_landmarks(image, face_mesh)

        if not landmarks:
            print("No person detected")
            continue

        # 모든 랜드마크 그리기
        for xy in landmarks[0]: # [0] is for the first person
            cv2.circle(image, xy, 2, (255, 255, 255), cv2.FILLED)

        # 특정 랜드마크만 그리기
        cv2.circle(image, landmarks[0][159], 2, (255, 255, 255), cv2.FILLED)
        cv2.circle(image, landmarks[0][145], 2, (255, 255, 255), cv2.FILLED)

        cv2.imshow("MediaPipe Face Mesh", image)
        if cv2.waitKey(5) == ord('q'):
            break

cap.release()
cv2.destroyAllWindows()

### [실습] 윙크 감지

힌트: 두 점 사이의 거리(Euclidean distance) 구하는 방법은 수학 시간에 배운 그대로 적용하면 됩니다. 검색해도 많이 나옵니다.

In [3]:
import cv2
import mediapipe as mp

def find_landmarks(img_bgr, face_mesh):
    """Normalized image space to pixel space"""

    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    results = face_mesh.process(img_rgb)
    landmarks = []
    if results.multi_face_landmarks:
        for person in results.multi_face_landmarks:
            face = []
            for id, lm in enumerate(person.landmark):
                ih, iw, _ = img_bgr.shape
                x, y = int(lm.x * iw), int(lm.y * ih)
                face.append([x, y])
            landmarks.append(face)
    return landmarks

#cap = cv2.VideoCapture("face_wink_sample.mp4")
cap = cv2.VideoCapture(0)

with mp.solutions.face_mesh.FaceMesh(
    max_num_faces=1,
    refine_landmarks=True,
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5,
) as face_mesh:

    while cap.isOpened():

        success, image = cap.read()
        if not success:            
            break

        landmarks = find_landmarks(image, face_mesh)

        if not landmarks:
            print("No person detected")
            continue

        cv2.circle(image, landmarks[0][159], 2, (255, 255, 255), cv2.FILLED)
        cv2.circle(image, landmarks[0][145], 2, (255, 255, 255), cv2.FILLED)

        if ((landmarks[0][159][0] - landmarks[0][145][0])**2 + (landmarks[0][159][1] - landmarks[0][145][1])**2 < 10):
            cv2.putText(image, "!", (int(500), int(100)), cv2.FONT_HERSHEY_SIMPLEX, 3, (0, 255, 0), 10)

        cv2.imshow("MediaPipe Face Mesh", image)
        if cv2.waitKey(5) == ord('q'):
            break

cap.release()
cv2.destroyAllWindows()

In [2]:
# 정답

import cv2
import mediapipe as mp
import numpy as np

def find_landmarks(img_bgr, face_mesh):
    """Normalized image space to pixel space"""

    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    results = face_mesh.process(img_rgb)
    landmarks = []
    if results.multi_face_landmarks:
        for person in results.multi_face_landmarks:
            face = []
            for id, lm in enumerate(person.landmark):
                ih, iw, _ = img_bgr.shape
                x, y = int(lm.x * iw), int(lm.y * ih)
                face.append([x, y])
            landmarks.append(face)
    return landmarks

cap = cv2.VideoCapture("face_wink_sample.mp4")
#cap = cv2.VideoCapture(0)

with mp.solutions.face_mesh.FaceMesh(
    max_num_faces=1,
    refine_landmarks=True,
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5,
) as face_mesh:

    while cap.isOpened():

        success, image = cap.read()
        if not success:            
            break

        landmarks = find_landmarks(image, face_mesh)

        if landmarks:
            l1 = np.array(landmarks[0][159])
            l2 = np.array(landmarks[0][145])
            sqr_dist = np.sum(np.square(l1-l2))
            if sqr_dist < 10:
                font = cv2.FONT_HERSHEY_SIMPLEX
                frame = cv2.putText(image, "!", (100, 200), font, 5, (255, 255, 255), 2, cv2.LINE_AA)
            
            cv2.circle(image, landmarks[0][159], 2, (255, 255, 255), cv2.FILLED)
            cv2.circle(image, landmarks[0][145], 2, (255, 255, 255), cv2.FILLED)

        cv2.imshow("MediaPipe Face Mesh", image)
        if cv2.waitKey(5) == ord('q'):
            break

cap.release()
cv2.destroyAllWindows()

### [선택/도전 문제] 윙크하면 하트 그리기
하트는 heart.jpg와 heart_mask.jpg를 사용하세요.

In [ ]:
# 정답

import cv2
import mediapipe as mp
import numpy as np

def find_landmarks(img_bgr, face_mesh):
    """Normalized image space to pixel space"""

    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    results = face_mesh.process(img_rgb)
    landmarks = []
    if results.multi_face_landmarks:
        for person in results.multi_face_landmarks:
            face = []
            for id, lm in enumerate(person.landmark):
                ih, iw, _ = img_bgr.shape
                x, y = int(lm.x * iw), int(lm.y * ih)
                face.append([x, y])
            landmarks.append(face)
    return landmarks

cap = cv2.VideoCapture("face_wink_sample.mp4")
#cap = cv2.VideoCapture(0)

with mp.solutions.face_mesh.FaceMesh(
    max_num_faces=1,
    refine_landmarks=True,
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5,
) as face_mesh:

    while cap.isOpened():

        success, image = cap.read()
        if not success:            
            break

        landmarks = find_landmarks(image, face_mesh)

        if landmarks:
            l1 = np.array(landmarks[0][159])
            l2 = np.array(landmarks[0][145])
            sqr_dist = np.sum(np.square(l1-l2))
            if sqr_dist < 10:
                #이 부분만 하트 나오게 변경하면 될듯
            

        cv2.imshow("MediaPipe Face Mesh", image)
        if cv2.waitKey(5) == ord('q'):
            break

cap.release()
cv2.destroyAllWindows()

In [1]:
#구현 예시

import cv2
import mediapipe as mp
import numpy as np
def find_landmarks(img_bgr, face_mesh):
    """Normalized image space to pixel space"""
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    results = face_mesh.process(img_rgb)
    landmarks = []
    if results.multi_face_landmarks:
        for person in results.multi_face_landmarks:
            face = []
            for id, lm in enumerate(person.landmark):
                ih, iw, _ = img_bgr.shape
                x, y = int(lm.x * iw), int(lm.y * ih)
                face.append([x, y])
            landmarks.append(face)
    return landmarks

def draw_sprite(img, spr, mask, center):
    h, w, _ = spr.shape
    x, y = center[0] - h//2, center[1] - w//2
    if any((x < 0, y < 0, y+h >= img.shape[0], w+h >= img.shape[1])):
        return img
    # Not safe near boundaries
    image[y:y+h, x:x+w] = image[y:y+h, x:x+w] * (1.0 - mask) + heart * heart_mask
    return image.astype(np.uint8)
heart = cv2.imread("heart.jpg")
heart_mask = cv2.imread("heart_mask.jpg")
heart = cv2.resize(heart, (50, 50))
heart_mask = cv2.resize(heart_mask, (50, 50)) / 255.0

#cap = cv2.VideoCapture("face_wink_sample.mp4")
cap = cv2.VideoCapture(0)

with mp.solutions.face_mesh.FaceMesh(
    max_num_faces=1,
    refine_landmarks=True,
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5,
) as face_mesh:
    while cap.isOpened():
        success, image = cap.read()
        if not success:
            print("Ignoring empty camera frame.")
            break
        landmarks = find_landmarks(image, face_mesh)
        if not landmarks:
            print("No person detected")
            continue
        l1 = np.array(landmarks[0][159])
        l2 = np.array(landmarks[0][145])
        if abs(l1[0] - l2[0]) < 3 and abs(l1[1] - l2[1]) < 3: # 눈을 감았는지 검사
            xy = (l1 + l2) * 0.5
            image = draw_sprite(image, heart, heart_mask, xy.astype(np.int32))
        cv2.imshow("MediaPipe Face Mesh",  cv2.flip(image, 1)) #좌우반전
        if cv2.waitKey(5) == ord('q'):
            break

cap.release()
cv2.destroyAllWindows()

No person detected


[참고]

거리를 비교할 때 squared distance를 사용하는 이유는 sqrt() 사용을 피하기 위해서입니다. 간단한 예제에서는 상관 없지만 일반적으로 속도가 중요한 프로그램에서는 sqrt() 사용을 피합니다.